# task_21 Solution

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

base = Path("../../tasks/task_21/data/")

In [ ]:
properties = pd.read_csv(base / "properties.csv")
split = pd.read_csv(base / "split.csv")

df = properties.merge(split, on="property_id")
train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()

numeric_features = ["sqft", "bedrooms", "bathrooms", "year_built", "lot_acres",
                    "garage_spaces", "renovated", "distance_miles", "months_listed"]

Q1: Forward selection -- starting from sqft only, which single additional numeric feature gives the largest R-squared improvement on training data?

In [ ]:
y_train = train["sale_price"].values

# Baseline: sqft only
lr_base = LinearRegression()
lr_base.fit(train[["sqft"]], y_train)
r2_base = r2_score(y_train, lr_base.predict(train[["sqft"]]))

candidates = [f for f in numeric_features if f != "sqft"]
best_gain = -np.inf
best_feature = None

for feat in candidates:
    lr = LinearRegression()
    lr.fit(train[["sqft", feat]], y_train)
    r2 = r2_score(y_train, lr.predict(train[["sqft", feat]]))
    gain = r2 - r2_base
    if gain > best_gain:
        best_gain = gain
        best_feature = feat

q1 = best_feature
print(q1)

Q2: Which feature has the highest Variance Inflation Factor (VIF)?

In [ ]:
# VIF for feature j = 1 / (1 - R^2_j) where R^2_j comes from regressing feature j on all other features
vif_results = {}
X_train_all = train[numeric_features]

for feat in numeric_features:
    others = [f for f in numeric_features if f != feat]
    lr = LinearRegression()
    lr.fit(X_train_all[others], X_train_all[feat])
    r2 = r2_score(X_train_all[feat], lr.predict(X_train_all[others]))
    vif_results[feat] = 1.0 / (1.0 - r2)

q2 = max(vif_results, key=vif_results.get)
print(q2)

Q3: Test RMSE rounded to nearest $1000 using all numeric features

In [ ]:
lr_all = LinearRegression()
lr_all.fit(train[numeric_features], train["sale_price"])
y_pred_test = lr_all.predict(test[numeric_features])
rmse = np.sqrt(mean_squared_error(test["sale_price"], y_pred_test))
q3 = int(round(rmse, -3))
print(q3)

Q4: Which neighborhood has the highest average residual (actual minus predicted) when the model is fit WITHOUT neighborhood?

In [ ]:
# Model is already fit on numeric features only (no neighborhood) as lr_all
train_pred = lr_all.predict(train[numeric_features])
train_resid = train.copy()
train_resid["residual"] = train_resid["sale_price"] - train_pred

q4 = train_resid.groupby("neighborhood")["residual"].mean().sort_values(ascending=False).index[0]
print(q4)

Q5: Which feature's removal causes the smallest R-squared drop?

In [ ]:
# Full model R-squared on training data
r2_full = r2_score(train["sale_price"], lr_all.predict(train[numeric_features]))

drops = {}
for feat in numeric_features:
    reduced = [f for f in numeric_features if f != feat]
    lr_red = LinearRegression()
    lr_red.fit(train[reduced], train["sale_price"])
    r2_red = r2_score(train["sale_price"], lr_red.predict(train[reduced]))
    drops[feat] = r2_full - r2_red

# Smallest drop; break ties alphabetically
q5 = min(drops, key=lambda f: (drops[f], f))
print(q5)